# glbench vs llama.cpp — a benchmark-reporting comparison

This notebook is **not** primarily a race. Throughput is Section 2 and it is
one section of seven. The deliverable is a structured comparison of what each
tool *tells you about the run*: statistical rigor, hardware insight,
behavioral signal, and whether the tool surfaces its own limits.

Output: `BENCHMARK_COMPARISON.md` in the working directory, with raw terminal
output attached as an appendix.

### How the "does this tool report X?" tables are filled

Not by memory or by reading source. Each metric has a set of **search
patterns**, and those patterns are run against the *maximum-verbosity output
of both tools* — terminal text plus the flattened JSON each one emits. A row
says "not reported" only after the search misses, and the patterns it searched
are printed in the report so a reader can disagree with them.

The method can produce a false "not reported" if a tool names a metric
something the patterns do not anticipate. That is a real weakness of this
methodology and it is stated in the report rather than hidden.

### Deviations from a naive comparison, and why

| # | Deviation | Reason |
|---|---|---|
| **D1** | llama.cpp built with `-j2`, `-DCMAKE_CUDA_ARCHITECTURES=75-real`, `LLAMA_CURL=OFF`, ccache | `-j$(nproc)` OOMs: nvcc is RAM-hungry and Kaggle gives 4 vCPU. Without the arch pin, the build compiles every CUDA architecture. |
| **D2** | `llama-bench` gets a *synthetic* prompt length; only `llama-cli` gets the exact text | `llama-bench -p N` takes a token **count**, not text. The two tools cannot be given an identical prompt through `llama-bench` at all. This is a limit of the comparison and is reported as one. |
| **D3** | A CUDA failure on either side aborts the run | glproc-on-CPU versus llama.cpp-on-CUDA is not a comparison. There is no silent fallback. |
| **D4** | "Reported" vs "derivable" are different answers | `llama-bench -o json` emits raw per-repetition samples. A statistic it does not print but that a reader can compute from what it does print is recorded as *derivable*, not as absent. |

### Known and already fixed

glbench's decode ceiling was computed from **host** DDR bandwidth on CUDA runs
(390% of a ceiling 12x too low on a T4), and a `clamp(0.0, 1.0)` printed that
as "100% of peak". Fixed in `39b1575` on this branch. If it still appears in
this run, that is a note, not a discovery.


## Step 1a — Config

Edit the top block if needed; nothing below it should need touching.

In [ ]:
# ---- edit these if needed -------------------------------------------------
REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BRANCH   = "glbench-vs-llamacpp"   # the ceiling fix + this notebook live here
GH_TOKEN = ""                      # only if the repo is private

# The model. Reused for both tools -- downloaded once, never twice.
MODEL_REPO = "https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/main"
MODEL_FILE = "qwen2.5-0.5b-instruct-q4_k_m.gguf"

# Matched workload. Both tools get exactly these.
GEN_TOKENS = 128
WARMUP     = 3
ITERS      = 10

# llama.cpp build (D1). ggerganov/llama.cpp redirects here.
LLAMA_REPO = "https://github.com/ggml-org/llama.cpp"
CUDA_ARCH  = "75-real"   # Tesla T4 is sm_75. Pin it: "all" builds every arch.

# Pin the commit. Two reasons, both practical: a benchmark document that
# names no version is not reproducible, and an unpinned --depth 1 clone
# lands on whatever HEAD is today, which misses the binary cache every
# time upstream moves. Set to "" to track the default branch instead.
LLAMA_PIN  = "cd26896"

# ⛔ -j2 was measured OOMing overnight on this box. ggml-cuda's FlashAttention
# translation units are template monsters and each nvcc process can hold well
# over 8 GB; two at once does not fit alongside the Python kernel. -j1 is
# slower in wall clock and finishes, which is the faster of the two.
BUILD_JOBS = 1

# Turning this on compiles WITHOUT ggml's CUDA FlashAttention kernels
# (GGML_CUDA_FA=OFF). Those kernels are the bulk of both the compile time and
# the peak memory, so this is the big lever -- and it changes how llama.cpp
# runs, so it is off by default and Section 1 says so loudly when it is used.
FAST_BUILD = False
# --------------------------------------------------------------------------

import os, sys, re, json, time, glob, shutil, subprocess, urllib.request

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
os.makedirs(WORK, exist_ok=True)
REPO_DIR  = os.path.join(WORK, "gwenland-ai")
LLAMA_DIR = os.path.join(WORK, "llama.cpp")
OUT_DIR   = WORK

def sh(cmd, cwd=None, timeout=7200, env=None):
    # Run a command; return (rc, stdout, stderr). Never raises on rc != 0 --
    # a failing tool is data for the report, not a reason to lose the session.
    e = dict(os.environ)
    if env:
        e.update(env)
    try:
        p = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True,
                           timeout=timeout, env=e)
        return p.returncode, p.stdout, p.stderr
    except subprocess.TimeoutExpired:
        return 124, "", f"timed out after {timeout}s"
    except Exception as ex:
        return 125, "", f"{type(ex).__name__}: {ex}"

def have_internet(timeout=6):
    try:
        urllib.request.urlopen("https://github.com", timeout=timeout)
        return True
    except Exception:
        return False

print(f"working dir : {WORK}")
print(f"internet    : {'YES' if have_internet() else 'NO  <-- Settings -> Internet'}")

rc, out, _ = sh(["nvidia-smi", "--query-gpu=name,compute_cap,memory.total",
                 "--format=csv,noheader"], timeout=60)
GPU_LINE = out.strip() if rc == 0 else "no nvidia-smi"
GPU_COUNT = len([ln for ln in GPU_LINE.splitlines() if ln.strip()]) if rc == 0 else 0
print(f"gpus        : {GPU_COUNT}")
for ln in GPU_LINE.splitlines():
    print(f"              {ln}")

# ⛔ Kaggle hands out TWO T4s, and llama.cpp splits the model across every
# visible device by default -- an observed run showed compute buffers on both
# CUDA0 and CUDA1. glbench's glcuda runs on one. Left alone this compares one
# GPU against two, which is not a comparison of engines at all.
#
# Masking at the driver level is the one control that applies identically to
# both tools without either needing a flag for it, so neither can quietly
# disagree about what hardware it got.
#
# llama.cpp's multi-GPU support is a real capability that glbench lacks. It is
# credited in Section 7 rather than being allowed to contaminate Section 2.
if GPU_COUNT > 1:
    print(f"\n{GPU_COUNT} GPUs visible -- restricting BOTH tools to device 0 "
          f"so the throughput table compares engines, not device counts.")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
SINGLE_GPU_ENFORCED = True

# Captured for Section 1 so the report states the machine it ran on.
RUN_META = {
    "date_utc": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
    "gpu": GPU_LINE.replace("\n", " | "),
    "gpu_count": GPU_COUNT,
    "host": " ".join(os.uname()) if hasattr(os, "uname") else sys.platform,
}


## Step 1b — Repo and toolchain

Clones the branch carrying the ceiling fix. A stale clone from an earlier
session is refreshed rather than reused, because the fix is the whole reason
this branch exists.

In [ ]:
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    url = REPO_URL
    if GH_TOKEN:
        url = REPO_URL.replace("https://", f"https://{GH_TOKEN}@")
    rc, o, e = sh(["git", "clone", "--depth", "1", "--branch", BRANCH, url, REPO_DIR])
    print(o[-1500:] or e[-1500:])
else:
    sh(["git", "fetch", "--depth", "1", "origin", BRANCH], cwd=REPO_DIR)
    sh(["git", "checkout", BRANCH], cwd=REPO_DIR)
    sh(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_DIR)
    print("refreshed existing clone")

rc, o, _ = sh(["git", "log", "--oneline", "-1"], cwd=REPO_DIR)
GL_COMMIT = o.strip()
print("glbench commit :", GL_COMMIT)

# The ceiling fix must be present, or Section 4 reports a bug as a feature.
CEILING_FIX = os.path.exists(os.path.join(REPO_DIR, "glbench/src/analysis/ceiling.rs")) and \
    "bandwidth_for_run" in open(os.path.join(REPO_DIR, "glbench/src/analysis/ceiling.rs"),
                                encoding="utf-8", errors="replace").read()
print("ceiling fix    :", "present" if CEILING_FIX else "MISSING -- wrong branch?")

if shutil.which("cargo") is None:
    os.system("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | "
              "sh -s -- -y --default-toolchain stable --profile minimal")
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
rc, o, _ = sh(["cargo", "--version"], timeout=120)
print("cargo          :", o.strip())


## Step 1c — Model, downloaded once

Both tools read the same file. Nothing is downloaded twice.

In [ ]:
MODEL_PATH = None
for cand in [os.path.join(WORK, MODEL_FILE), os.path.join(REPO_DIR, MODEL_FILE)]:
    if os.path.exists(cand) and os.path.getsize(cand) > 10_000_000:
        MODEL_PATH = cand
        break

# Kaggle datasets, if the user attached one instead.
if MODEL_PATH is None and os.path.isdir("/kaggle/input"):
    for root, _, files in os.walk("/kaggle/input"):
        for f in files:
            if f.lower().endswith(".gguf") and "0.5b" in f.lower():
                MODEL_PATH = os.path.join(root, f)
                break
        if MODEL_PATH:
            break

if MODEL_PATH is None:
    dest = os.path.join(WORK, MODEL_FILE)
    url = f"{MODEL_REPO}/{MODEL_FILE}"
    print(f"downloading {url}")
    rc, o, e = sh(["curl", "-fL", "--retry", "3", "-o", dest, url], timeout=1800)
    if rc == 0 and os.path.exists(dest) and os.path.getsize(dest) > 10_000_000:
        MODEL_PATH = dest
    else:
        print("download failed:", e[-500:])

if MODEL_PATH:
    MODEL_BYTES = os.path.getsize(MODEL_PATH)
    print(f"model : {MODEL_PATH}")
    print(f"size  : {MODEL_BYTES/1e9:.3f} GB ({MODEL_BYTES} bytes)")
else:
    raise RuntimeError("no model -- everything downstream compares nothing")


## Step 2 — Build llama.cpp with CUDA (D1)

> ### Read this before running the notebook overnight
>
> Upstream publishes **no Linux CUDA binary** — the prebuilt CUDA archives are
> Windows-only, and the Linux ones are CPU, Vulkan, SYCL and OpenVINO. So this
> has to be compiled, and on a Kaggle box it is by far the most expensive cell
> here: **28.2 minutes measured**, and `-j2` was observed running out of memory
> when left overnight.
>
> Three things follow from that, and they are why this cell looks the way it
> does:
>
> 1. **`BUILD_JOBS = 1`.** ggml-cuda's FlashAttention translation units are
>    template monsters; a single nvcc process can hold well over 8 GB and two
>    at once do not fit beside the Python kernel. `-j1` has a worse wall clock
>    and finishes, which makes it the faster of the two.
> 2. **The binaries are cached.** After a successful build they are copied to
>    `/kaggle/working/llama-bin-cache` with a manifest recording the commit and
>    the exact flags. **Save the notebook version** so `/kaggle/working`
>    persists, and every later run skips the build entirely. ccache is pointed
>    there too — `~/.ccache` does not survive a session.
> 3. **Run it as a saved version, not an interactive tab.** *Save Version →
>    Run All* executes in batch: it does not need the browser open and does not
>    die when you close the laptop. An interactive session left overnight is
>    the scenario that already failed once.
>
> The cache manifest deliberately refuses to reuse binaries built with
> different flags. Binaries compiled with `GGML_CUDA_NO_VMM` or without
> FlashAttention are not interchangeable with a stock build, and quietly
> reusing them would put a handicap into Section 2 that nothing in the report
> could account for.
>
> If you need the build to be much cheaper and can accept a disclosed
> handicap, set `FAST_BUILD = True` in the config cell. That compiles with
> `GGML_CUDA_FA=OFF`, dropping the kernels that dominate both compile time and
> peak memory — and Section 1 will say, in the report, that llama.cpp ran
> without FlashAttention.

A pinned `sm_75` is likewise not cosmetic: an unpinned arch list compiles every
CUDA architecture ever shipped.

### The `CUDA::cuda_driver` failure, and why it is handled in tiers

Observed on this image at llama.cpp `cd26896`:

```
CMake Error at ggml/src/ggml-cuda/CMakeLists.txt:182 (target_link_libraries):
  Target "ggml-cuda" links to: CUDA::cuda_driver
  but the target was not found.
```

CMake found the toolkit — includes, nvcc, version 12.8.93 — but
`FindCUDAToolkit` only creates the `CUDA::cuda_driver` target if it also finds
`libcuda.so`, and a CUDA image can ship the runtime without that development
symlink. So the cell locates `libcuda` itself and hands CMake the path.

Upstream does offer an escape hatch, in its own words:

```cmake
if (GGML_CUDA_NO_VMM)
    # No VMM requested, no need to link directly with the cuda driver lib
else()
    target_link_libraries(ggml-cuda PRIVATE CUDA::cuda_driver)
endif()
```

That flag would configure immediately — and it disables ggml's CUDA
virtual-memory pool allocator, which is a **performance-relevant change to the
tool being measured**. Reaching for it first would quietly handicap llama.cpp
in the throughput table and make glbench look better for a reason that has
nothing to do with either engine. It is therefore the last resort, and taking
it writes a warning into Section 1.

### `llama-cli` lives under the server gate now

Measured on the first attempt: 28.2 minutes of compiling, then

```
[100%] Built target llama-bench
gmake: *** No rule to make target 'llama-cli'.  Stop.
```

`llama-bench` built; `llama-cli` had no target at all. The cause was a flag
passed to save build time. In `tools/CMakeLists.txt` at this commit:

```cmake
if (LLAMA_BUILD_SERVER)
    add_subdirectory(ui)
    add_subdirectory(cli)      # llama-cli — moved here from tools/main
    add_subdirectory(server)
endif()
```

`ui`, `cli` and `server` share one gate, so `-DLLAMA_BUILD_SERVER=OFF` removes
the CLI. `--target llama-bench llama-cli` still limits what actually compiles,
so turning the gate on costs configure time rather than build time.

**This cell raises if the resulting binary has no CUDA backend (D3)** — but a
missing `llama-cli` alone does not abort the run. `llama-bench` carries the
throughput comparison; the CLI only feeds the behavioral section and the
tokenizer cross-check, and those degrade to "unavailable" with a note in
Section 1. Throwing away a working Section 2 over a missing Section 5 would be
the wrong trade.

In [ ]:
if not os.path.isdir(os.path.join(LLAMA_DIR, ".git")):
    rc, o, e = sh(["git", "clone", "--depth", "1", LLAMA_REPO, LLAMA_DIR], timeout=1800)
    print((o or e)[-800:])

if LLAMA_PIN:
    rc, o, _ = sh(["git", "rev-parse", "--short", "HEAD"], cwd=LLAMA_DIR)
    if not o.strip().startswith(LLAMA_PIN[:7]):
        # A shallow clone has only the tip, so the pinned commit has to be
        # fetched explicitly before it can be checked out.
        rc, o, e = sh(["git", "fetch", "--depth", "1", "origin", LLAMA_PIN],
                      cwd=LLAMA_DIR, timeout=900)
        rc2, _, e2 = sh(["git", "checkout", "-q", LLAMA_PIN], cwd=LLAMA_DIR)
        if rc2 != 0:
            print(f"could not pin to {LLAMA_PIN} ({(e or e2).strip()[:200]}); "
                  f"staying on the default branch tip")

rc, o, _ = sh(["git", "rev-parse", "--short", "HEAD"], cwd=LLAMA_DIR)
LLAMA_COMMIT = o.strip()
print("llama.cpp commit:", LLAMA_COMMIT,
      f"(pinned to {LLAMA_PIN})" if LLAMA_PIN else "(tracking default branch)")

os.system("which ccache >/dev/null 2>&1 || (apt-get -qq install -y ccache >/dev/null 2>&1)")
USE_CCACHE = shutil.which("ccache") is not None
# ~/.ccache does NOT survive a Kaggle session. /kaggle/working does, once the
# notebook is saved, so point ccache there or every session pays full price.
os.environ["CCACHE_DIR"] = os.path.join(WORK, ".ccache")
os.environ["CCACHE_MAXSIZE"] = "5G"
os.makedirs(os.environ["CCACHE_DIR"], exist_ok=True)
print("ccache          :", "yes" if USE_CCACHE else "no (build will be slower)")
print("ccache dir      :", os.environ["CCACHE_DIR"])

LLAMA_BIN_DIR = os.path.join(LLAMA_DIR, "build", "bin")
LLAMA_BENCH = os.path.join(LLAMA_BIN_DIR, "llama-bench")
LLAMA_CLI   = os.path.join(LLAMA_BIN_DIR, "llama-cli")

# ---- the binary cache: pay for this build once, not once per session -------
#
# A 28-minute CUDA build that has to be repeated every session is the reason
# this notebook kept dying overnight. Finished binaries are copied into
# /kaggle/working, which survives a saved session, and reused next time.
#
# The manifest is what makes reuse safe: binaries built with GGML_CUDA_NO_VMM
# or without FlashAttention are NOT interchangeable with a stock build, and
# silently reusing them would put a handicap into Section 2 that nothing in
# the report could explain. Reuse only on an exact match of commit and flags.
BIN_CACHE = os.path.join(WORK, "llama-bin-cache")
CACHE_MANIFEST = os.path.join(BIN_CACHE, "manifest.json")

def cache_key():
    return {"commit": LLAMA_COMMIT, "arch": CUDA_ARCH, "fast_build": FAST_BUILD,
            "strategy": None}   # strategy filled after configure

def try_restore_cache():
    if not os.path.exists(CACHE_MANIFEST):
        return None
    try:
        man = json.load(open(CACHE_MANIFEST, encoding="utf-8"))
    except Exception:
        return None
    want = cache_key()
    if man.get("commit") != want["commit"] or man.get("arch") != want["arch"] \
            or man.get("fast_build") != want["fast_build"]:
        print(f"binary cache present but does not match "
              f"(cached: {man.get('commit')}/fa_off={man.get('fast_build')}); ignoring")
        return None
    ok = True
    os.makedirs(LLAMA_BIN_DIR, exist_ok=True)
    for name in ("llama-bench", "llama-cli"):
        src = os.path.join(BIN_CACHE, name)
        if os.path.exists(src):
            dst = os.path.join(LLAMA_BIN_DIR, name)
            shutil.copy2(src, dst)
            os.chmod(dst, 0o755)
        elif name == "llama-bench":
            ok = False
    # Shared libraries the tools link against travel with them.
    for so in glob.glob(os.path.join(BIN_CACHE, "*.so*")):
        shutil.copy2(so, os.path.join(LLAMA_BIN_DIR, os.path.basename(so)))
    return man if ok else None

def save_cache(strategy):
    os.makedirs(BIN_CACHE, exist_ok=True)
    saved = []
    for name in ("llama-bench", "llama-cli"):
        src = os.path.join(LLAMA_BIN_DIR, name)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(BIN_CACHE, name))
            saved.append(name)
    for so in glob.glob(os.path.join(LLAMA_BIN_DIR, "*.so*")):
        shutil.copy2(so, os.path.join(BIN_CACHE, os.path.basename(so)))
    man = cache_key()
    man["strategy"] = strategy
    man["saved"] = saved
    json.dump(man, open(CACHE_MANIFEST, "w", encoding="utf-8"), indent=1)
    print(f"binary cache written to {BIN_CACHE}: {saved}")
    print("  Save this notebook version so /kaggle/working persists, then the "
          "next run skips the build entirely.")

RESTORED = try_restore_cache()

# Cold vs warm, decided before the timer -- a rebuilt tree relinks in seconds
# and that number is not a build time.
LLAMA_COLD = not os.path.exists(LLAMA_BENCH)

# ggml-cuda links CUDA::cuda_driver, a target FindCUDAToolkit only creates if
# it can find libcuda.so. A CUDA image can ship the toolkit without the
# development symlink, so the toolkit is "found" and the target still is not:
#
#   CMake Error at ggml/src/ggml-cuda/CMakeLists.txt:182
#     Target "ggml-cuda" links to: CUDA::cuda_driver
#     but the target was not found.
#
# Find the library ourselves and hand CMake the path.
LIBCUDA_CANDIDATES = []

# 1. The dynamic linker cache. Authoritative and path-independent -- a glob
#    list only finds libcuda where you already guessed it might be, and on
#    this image the first version of that guess found nothing at all.
rc, o, _ = sh(["ldconfig", "-p"], timeout=120)
for line in o.splitlines():
    if "libcuda.so" in line and "=>" in line:
        p = line.split("=>")[-1].strip()
        if os.path.exists(p):
            LIBCUDA_CANDIDATES.append(p)

# 2. Toolkit stubs, which ldconfig never lists because they are link-time only.
for pat in ["/usr/local/cuda*/lib64/stubs/libcuda.so",
            "/usr/local/cuda*/targets/*/lib/stubs/libcuda.so",
            "/usr/local/cuda/lib64/stubs/libcuda.so"]:
    LIBCUDA_CANDIDATES += sorted(glob.glob(pat))

# 3. A bounded sweep, so a failure produces evidence rather than a shrug.
rc, o, _ = sh(["bash", "-lc",
               "find /usr /opt /lib /lib64 -maxdepth 6 -name 'libcuda.so*' "
               "2>/dev/null | head -40"], timeout=300)
LIBCUDA_CANDIDATES += [ln.strip() for ln in o.splitlines() if ln.strip()]

# Dedupe, preferring the link-time name `libcuda.so` over a runtime `.so.N`.
seen, ordered = set(), []
for p in LIBCUDA_CANDIDATES:
    if p not in seen:
        seen.add(p)
        ordered.append(p)
ordered.sort(key=lambda p: (os.path.basename(p) != "libcuda.so", len(p)))
LIBCUDA_CANDIDATES = ordered
print("libcuda candidates:")
for p in LIBCUDA_CANDIDATES:
    print("   ", p)
if not LIBCUDA_CANDIDATES:
    print("    NONE FOUND -- the NO_VMM fallback will be needed")

LIBCUDA = LIBCUDA_CANDIDATES[0] if LIBCUDA_CANDIDATES else None

# find_library looks for the bare `libcuda.so` name. If all we have is a
# runtime `libcuda.so.1`, give CMake a directory where that name exists.
LIBCUDA_SHIM = None
if LIBCUDA and os.path.basename(LIBCUDA) != "libcuda.so":
    LIBCUDA_SHIM = os.path.join(WORK, "cuda-link-shim")
    os.makedirs(LIBCUDA_SHIM, exist_ok=True)
    link = os.path.join(LIBCUDA_SHIM, "libcuda.so")
    if not os.path.exists(link):
        try:
            os.symlink(LIBCUDA, link)
        except OSError as ex:
            print(f"  could not create link shim: {ex}")
            LIBCUDA_SHIM = None
    if LIBCUDA_SHIM:
        print(f"  link shim: {link} -> {LIBCUDA}")

BASE_CFG = ["cmake", "-B", "build",
            "-DGGML_CUDA=ON",
            "-DCMAKE_BUILD_TYPE=Release",
            f"-DCMAKE_CUDA_ARCHITECTURES={CUDA_ARCH}",
            "-DLLAMA_CURL=OFF",
            "-DLLAMA_BUILD_TESTS=OFF",
            "-DLLAMA_BUILD_EXAMPLES=OFF",
            # ⛔ NOT OFF, however much it looks like a saving. In tools/CMakeLists.txt:
            #
            #     if (LLAMA_BUILD_SERVER)
            #         add_subdirectory(ui)
            #         add_subdirectory(cli)
            #         add_subdirectory(server)
            #     endif()
            #
            # `llama-cli` moved under the server gate (tools/main -> tools/cli),
            # so -DLLAMA_BUILD_SERVER=OFF deletes the CLI target and the build
            # ends with "No rule to make target 'llama-cli'" after 28 minutes of
            # compiling everything else. `--target` still limits what is
            # actually compiled, so turning this ON costs configure time, not
            # build time.
            "-DLLAMA_BUILD_SERVER=ON",
            # Multi-GPU collectives. Safe here only because the config cell
            # pins CUDA_VISIBLE_DEVICES=0 for both tools -- an earlier version
            # of this comment claimed no second card was in play, which was
            # simply wrong: Kaggle gives two T4s and llama.cpp was observed
            # using both. With one device enforced, this cannot change
            # throughput, unlike NO_VMM or FA.
            "-DGGML_CUDA_NCCL=OFF"]
if FAST_BUILD:
    BASE_CFG.append("-DGGML_CUDA_FA=OFF")
if USE_CCACHE:
    BASE_CFG += ["-DCMAKE_C_COMPILER_LAUNCHER=ccache",
                 "-DCMAKE_CXX_COMPILER_LAUNCHER=ccache",
                 "-DCMAKE_CUDA_COMPILER_LAUNCHER=ccache"]

# Ordered by how little they change about llama.cpp. GGML_CUDA_NO_VMM is last
# on purpose: upstream's own comment says it exists to avoid linking the
# driver lib, so it WILL configure -- but it also turns off the VMM pool
# allocator, which is a performance-relevant change to the tool being
# measured. Taking it silently would quietly handicap llama.cpp in Section 2.
ATTEMPTS = []
if LIBCUDA:
    _dirs = [d for d in [LIBCUDA_SHIM, os.path.dirname(LIBCUDA)] if d]
    ATTEMPTS.append((f"libcuda pinned to {LIBCUDA}",
                     [f"-DCUDA_cuda_driver_LIBRARY={LIBCUDA}",
                      f"-DCUDA_CUDA_LIBRARY={LIBCUDA}",
                      "-DCMAKE_LIBRARY_PATH=" + ";".join(_dirs)]))
ATTEMPTS.append(("stock configuration", []))
ATTEMPTS.append(("GGML_CUDA_NO_VMM=ON -- VMM pool allocator DISABLED",
                 ["-DGGML_CUDA_NO_VMM=ON"]))

LLAMA_BUILD_NOTES = []
CONFIG_OK, CONFIG_STRATEGY = False, None
t0 = time.time()

if RESTORED:
    CONFIG_OK = True
    CONFIG_STRATEGY = RESTORED.get("strategy") or "restored from binary cache"
    ATTEMPTS = []
    print(f"reusing cached binaries: {RESTORED.get('saved')} "
          f"(built at {RESTORED.get('commit')}, strategy: {CONFIG_STRATEGY})")
    print("no build needed")

for label, extra in ATTEMPTS:
    # A poisoned cache makes the next attempt re-fail for the previous
    # attempt's reason. Clear it, not the whole tree -- ccache keeps the
    # object files either way.
    for stale in ["build/CMakeCache.txt", "build/CMakeFiles"]:
        p = os.path.join(LLAMA_DIR, stale)
        if os.path.isdir(p):
            shutil.rmtree(p, ignore_errors=True)
        elif os.path.exists(p):
            os.remove(p)

    print(f"\n--- configure: {label} ---")
    rc, o, e = sh(BASE_CFG + extra, cwd=LLAMA_DIR, timeout=1800)
    if rc == 0:
        CONFIG_OK, CONFIG_STRATEGY = True, label
        print("configure OK")
        break
    tail = [ln for ln in (o + e).splitlines() if "error" in ln.lower()][-6:]
    print("failed:", " | ".join(tail) if tail else (o + e)[-600:])

if not CONFIG_OK:
    print("\nALL CONFIGURE ATTEMPTS FAILED")
else:
    if "NO_VMM" in CONFIG_STRATEGY:
        LLAMA_BUILD_NOTES.append(
            "Built with `GGML_CUDA_NO_VMM=ON`. The stock configuration could "
            "not link `CUDA::cuda_driver` on this image. This disables ggml's "
            "CUDA virtual-memory pool allocator, so llama.cpp's throughput "
            "here may be BELOW what a stock build achieves. Any decode "
            "advantage glbench shows in Section 2 must be read with this in "
            "mind.")
    elif LIBCUDA and "pinned" in CONFIG_STRATEGY:
        LLAMA_BUILD_NOTES.append(
            f"`CUDA::cuda_driver` was not auto-discovered; libcuda was pinned "
            f"to `{LIBCUDA}`. This changes nothing about the code that runs.")

    if FAST_BUILD:
        LLAMA_BUILD_NOTES.append(
            "Built with `GGML_CUDA_FA=OFF`: ggml's CUDA FlashAttention kernels "
            "were not compiled, so llama.cpp runs without them. This was done "
            "to fit the build into the machine, not because FA is irrelevant, "
            "and it may depress llama.cpp's numbers below a stock build.")

    rc, o, e = sh(["cmake", "--build", "build", "--config", "Release",
                   "-j", str(BUILD_JOBS),
                   "--target", "llama-bench", "llama-cli"],
                  cwd=LLAMA_DIR, timeout=21600)
    if rc != 0:
        print("BUILD FAILED"); print((o + e)[-6000:])
    else:
        save_cache(CONFIG_STRATEGY)

LLAMA_BUILD_SECS = time.time() - t0
print(f"\nllama.cpp build: {LLAMA_BUILD_SECS/60:.1f} min "
      f"[{'COLD' if LLAMA_COLD else 'WARM - a relink, not a build time'}]")
print(f"strategy: {CONFIG_STRATEGY}")
for n in LLAMA_BUILD_NOTES:
    print("NOTE:", n)


### CUDA verification — hard gate (D3)

A build can succeed and still produce a CPU-only binary. The check is not "did
cmake print CUDA", it is "does a real inference run report a CUDA device".

In [ ]:
# llama-bench is the throughput comparison; without it there is nothing to
# compare. llama-cli only feeds the behavioral section and the tokenizer
# cross-check, so its absence degrades the report rather than voiding it --
# aborting the whole run over it would throw away a working Section 2.
if not os.path.exists(LLAMA_BENCH):
    raise RuntimeError(
        f"llama-bench missing ({LLAMA_BENCH}). Read the build log above. "
        "Not continuing: there is nothing to compare against."
    )

HAVE_CLI = os.path.exists(LLAMA_CLI)
if not HAVE_CLI:
    print("WARNING: llama-cli was not built. Sections 5 and the tokenizer "
          "cross-check will be reported as unavailable, and Section 1 will "
          "say so.")

# The only trustworthy probe is an actual run. A one-token bench loads the
# model through the same path the real measurement will use.
rc, smoke_out, smoke_err = sh([LLAMA_BENCH, "-m", MODEL_PATH,
                               "-p", "1", "-n", "1", "-r", "1", "-ngl", "99"],
                              timeout=900)
SMOKE = smoke_out + "\n" + smoke_err
CUDA_MARKERS = ["ggml_cuda_init", "CUDA0", "using CUDA", "found 1 CUDA", "found 2 CUDA"]
LLAMA_CUDA = any(m in SMOKE for m in CUDA_MARKERS)

print(SMOKE[-2500:])
print("\nCUDA backend detected:", LLAMA_CUDA)

if not LLAMA_CUDA:
    raise RuntimeError(
        "llama.cpp is running WITHOUT CUDA.\n"
        "Refusing to continue (D3): a CPU llama.cpp against a CUDA glbench "
        "produces a throughput table that looks like a result and is not one.\n"
        "Fix the build (check nvcc is on PATH and GGML_CUDA=ON took effect), "
        "or state the CPU fallback explicitly in Section 1 and re-run with this "
        "guard removed on purpose."
    )


## Step 3 — The prompt, extracted rather than retyped

`default_prompt()` is parsed out of the glbench source and reconstructed, so
the text used here cannot drift from the text glbench actually uses. The
**token count** is not guessed and not counted by a Python tokenizer either —
it comes from `iterations[0].prompt_tokens` in glbench's own JSON, which is
the count its real tokenizer produced.

If a previous run's JSON is lying around it is read here as an early value.
The authoritative count is taken from *this* notebook's glbench run in Step 4;
if the two disagree, that is printed rather than silently resolved.

In [ ]:
main_rs = os.path.join(REPO_DIR, "glbench", "src", "main.rs")
src = open(main_rs, encoding="utf-8", errors="replace").read()

body = re.search(r"fn default_prompt\(\)\s*->\s*String\s*\{(.*?)\n\}", src, re.S)
if not body:
    raise RuntimeError("default_prompt() not found -- source layout changed")
body = body.group(1)

lit = re.search(r'let\s+base\s*=\s*"(.*?)"\s*;', body, re.S)
rep = re.search(r"base\.repeat\((\d+)\)", body)
if not (lit and rep):
    raise RuntimeError(f"could not parse default_prompt():\n{body}")

# Rust's line-continuation backslash eats the newline AND the leading
# whitespace of the next line. Reproduce that, or the prompt gains spaces
# that glbench never sends.
base = re.sub(r"\\\n\s*", "", lit.group(1))
PROMPT_TEXT = (base * int(rep.group(1))).strip()

print(f"base sentence : {base!r}")
print(f"repeat        : {rep.group(1)}x, then trimmed")
print(f"chars         : {len(PROMPT_TEXT)}")
print(f"\nprompt:\n{PROMPT_TEXT[:300]}...")

# Early (non-authoritative) token count from any prior glbench archive.
PROMPT_TOKENS_EARLY = None
for cand in [os.path.join(REPO_DIR, "phase1.json"), os.path.join(WORK, "phase1.json"),
             os.path.join(WORK, "glbench_result.json")]:
    if os.path.exists(cand):
        try:
            j = json.load(open(cand, encoding="utf-8"))
            its = (j.get("measurements") or {}).get("iterations") or []
            if its and its[0].get("prompt_tokens"):
                PROMPT_TOKENS_EARLY = int(its[0]["prompt_tokens"])
                print(f"\nearly token count from {os.path.basename(cand)}: {PROMPT_TOKENS_EARLY}")
                break
        except Exception as ex:
            print(f"  ({cand}: {type(ex).__name__})")

if PROMPT_TOKENS_EARLY is None:
    print("\nno prior archive -- the count comes from Step 4's own run")

PROMPT_TOKENS = None   # set authoritatively in Step 4


## Step 4 — glbench (reference run)

Reuses `target/release/glbench` if this session already built it; builds it
otherwise. `--engine glcuda` is required, not preferred: a fallback to glproc
would silently turn this into a CPU-vs-GPU comparison (D3).

In [ ]:
GL_BIN = os.path.join(REPO_DIR, "target", "release", "glbench")
GL_BUILD_SECS, GL_COLD = 0.0, not os.path.exists(GL_BIN)

if GL_COLD:
    print("building glbench (release, this takes a while)...")
    t0 = time.time()
    rc, o, e = sh(["cargo", "build", "--release", "-p", "glbench"],
                  cwd=REPO_DIR, timeout=7200)
    GL_BUILD_SECS = time.time() - t0
    if rc != 0:
        print((o + e)[-5000:])
        raise RuntimeError("glbench build failed")
    print(f"built in {GL_BUILD_SECS/60:.1f} min")
else:
    print("reusing existing glbench binary")

GL_JSON_PATH = os.path.join(OUT_DIR, "glbench_result.json")
gl_cmd = [GL_BIN, "run",
          "--engine", "glcuda",
          "--model", MODEL_PATH,
          "--tokens", str(GEN_TOKENS),
          "--warmup", str(WARMUP),
          "--iters", str(ITERS),
          "--out", GL_JSON_PATH]
print("\n$ " + " ".join(gl_cmd) + "\n")

t0 = time.time()
rc, GL_STDOUT, GL_STDERR = sh(gl_cmd, cwd=REPO_DIR, timeout=7200)
GL_WALL = time.time() - t0

open(os.path.join(OUT_DIR, "glbench_output.txt"), "w", encoding="utf-8").write(
    GL_STDOUT + "\n===== stderr =====\n" + GL_STDERR)
print(GL_STDOUT[-6000:])
if rc != 0:
    print("\n===== stderr =====\n" + GL_STDERR[-3000:])
    raise RuntimeError(
        f"glbench --engine glcuda failed (exit {rc}). Not falling back to "
        "glproc (D3): CPU-vs-CUDA is not the comparison this notebook claims "
        "to make."
    )

GL_JSON = json.load(open(GL_JSON_PATH, encoding="utf-8"))
_its = (GL_JSON.get("measurements") or {}).get("iterations") or []
if not _its or not _its[0].get("prompt_tokens"):
    raise RuntimeError("glbench JSON has no iterations[0].prompt_tokens -- "
                       "the token count must not be approximated")

PROMPT_TOKENS = int(_its[0]["prompt_tokens"])
print(f"\nAUTHORITATIVE prompt_tokens = {PROMPT_TOKENS}  (iterations[0], glbench tokenizer)")
if PROMPT_TOKENS_EARLY is not None and PROMPT_TOKENS_EARLY != PROMPT_TOKENS:
    print(f"  NOTE: a prior archive said {PROMPT_TOKENS_EARLY}. Different binary or "
          f"model. This run's number is the one used.")
print(f"glbench wall time: {GL_WALL/60:.1f} min")


## Step 5 — llama.cpp, matched (D2)

**D2, stated plainly:** `llama-bench -p N` takes a token *count* and generates
a synthetic prompt of that length. It cannot be handed the same text. So:

* `llama-bench` gets `-p PROMPT_TOKENS` — same **length**, synthetic content.
  This is the throughput comparison, and it is length-matched, not text-matched.
* `llama-cli` gets the exact prompt **text** with `--verbose-prompt`. This is
  where behavioral output and llama.cpp's own tokenization of the same string
  come from.

llama.cpp re-tokenizing the identical text is a free cross-check on glbench's
tokenizer. If the two counts differ, that is a finding, and it is reported.

In [ ]:
assert PROMPT_TOKENS, "Step 4 must run first -- the token count is not guessable"

# --- llama-bench: throughput, length-matched synthetic prompt (D2) ---------
LB_JSON_PATH = os.path.join(OUT_DIR, "llama_result.json")
lb_cmd = [LLAMA_BENCH, "-m", MODEL_PATH,
          "-p", str(PROMPT_TOKENS),
          "-n", str(GEN_TOKENS),
          "-r", str(ITERS),
          "-ngl", "99",
          "-v",
          "-o", "json",
          # A second format on stderr: the human-readable table llama.cpp is
          # known for, captured for the appendix without disturbing the JSON
          # that stdout carries.
          "-oe", "md"]
print("$ " + " ".join(lb_cmd) + "\n")

t0 = time.time()
rc, LB_STDOUT, LB_STDERR = sh(lb_cmd, timeout=7200)
LB_WALL = time.time() - t0

open(LB_JSON_PATH, "w", encoding="utf-8").write(LB_STDOUT)
open(os.path.join(OUT_DIR, "llama_output.txt"), "w", encoding="utf-8").write(
    LB_STDOUT + "\n===== stderr =====\n" + LB_STDERR)

LB_JSON = None
try:
    LB_JSON = json.loads(LB_STDOUT)
except Exception as ex:
    m = re.search(r"\[\s*\{.*\}\s*\]", LB_STDOUT, re.S)   # verbose logs can lead
    if m:
        try:
            LB_JSON = json.loads(m.group(0))
        except Exception:
            pass
    if LB_JSON is None:
        print(f"llama-bench JSON unparsable ({type(ex).__name__}); "
              f"tables fall back to the text output")

print(LB_STDERR[-2500:])
print(f"\nllama-bench: exit {rc}, {LB_WALL/60:.1f} min, "
      f"{len(LB_JSON) if LB_JSON else 0} JSON rows")

# --- everything else llama-bench is willing to say -------------------------
#
# The rule this notebook sets itself is that "not reported" may only be
# written after checking the tool's verbosity flags. llama-bench has a
# dedicated hardware-reporting command, and not running it would have let
# Section 4 mark device information absent for a tool that has a whole flag
# for printing it.
rc3, LB_DEVICES, e3 = sh([LLAMA_BENCH, "--list-devices"], timeout=600)
LB_DEVICES = (LB_DEVICES or "") + "\n" + (e3 or "")
print("\n$ llama-bench --list-devices")
print(LB_DEVICES.strip()[:1500])

# Captured separately and deliberately KEPT OUT of the search corpus: a usage
# block lists flags, not measurements. Searching it would let `--no-warmup`
# answer "does this tool report warmup?" with a yes, which is precisely the
# false positive the negation pass exists to prevent. It is used for one
# narrow, explicit question below.
_, LB_HELP, _e = sh([LLAMA_BENCH, "--help"], timeout=120)
LB_HELP = (LB_HELP or "") + "\n" + (_e or "")
LB_WARMS_UP = "--no-warmup" in LB_HELP
print(f"\nllama-bench warms up by default: {LB_WARMS_UP} "
      f"(inferred from the presence of --no-warmup)")

# --- llama-cli: the exact text, for behavior + its own tokenization --------
def lc_base(extra):
    return [LLAMA_CLI, "-m", MODEL_PATH,
            "-p", PROMPT_TEXT,
            "-n", str(GEN_TOKENS),
            "-ngl", "99",
            "--verbose-prompt",
            "-v"] + extra

# `-no-cnv` forces plain completion instead of chat mode, but the flag has
# been renamed across releases. Try it, and drop it if this build rejects it
# rather than losing the whole behavioral section to an argument-parse error.
if HAVE_CLI:
    print("\n$ llama-cli ... --verbose-prompt -v  (exact prompt text)\n")
    LC_CONV_MODE = "completion (-no-cnv)"
    rc2, LC_STDOUT, LC_STDERR = sh(lc_base(["-no-cnv"]), timeout=3600)
    if rc2 != 0 and re.search(r"(unknown|unrecognized|invalid) (argument|option)",
                              LC_STDERR or "", re.I):
        print("  this build does not accept -no-cnv; retrying without it")
        LC_CONV_MODE = "default mode (-no-cnv rejected by this build)"
        rc2, LC_STDOUT, LC_STDERR = sh(lc_base([]), timeout=3600)
    print((LC_STDERR or "")[-2500:])
    print(f"\nllama-cli: exit {rc2}, {LC_CONV_MODE}")
else:
    rc2, LC_STDOUT, LC_STDERR = 127, "", ""
    LC_CONV_MODE = "NOT RUN -- llama-cli was not built"
    print("\nskipping llama-cli: binary absent")

open(os.path.join(OUT_DIR, "llama_cli_verbose.txt"), "w", encoding="utf-8").write(
    LC_STDOUT + "\n===== stderr =====\n" + LC_STDERR)

# llama.cpp's own count of the identical string.
LC_HAY = LC_STDOUT + "\n" + LC_STDERR
LLAMA_PROMPT_TOKENS = None
for pat in [r"n_tokens\s*=\s*(\d+)", r"prompt.*?(\d+)\s+tokens",
            r"tokens?\s*in\s*prompt\s*[:=]\s*(\d+)"]:
    m = re.search(pat, LC_HAY, re.I)
    if m:
        LLAMA_PROMPT_TOKENS = int(m.group(1))
        break

print(f"\nprompt tokens -- glbench: {PROMPT_TOKENS}  llama.cpp: "
      f"{LLAMA_PROMPT_TOKENS if LLAMA_PROMPT_TOKENS is not None else 'not found in output'}")
if LLAMA_PROMPT_TOKENS is not None and LLAMA_PROMPT_TOKENS != PROMPT_TOKENS:
    print("  ^ the two tokenizers disagree on identical text. Reported in Section 1.")


## Step 6 — Build the comparison report

Every "reported / not reported" cell below is the result of a search over that
tool's own maximum-verbosity output. Nothing is filled from memory.

In [ ]:
def flatten(obj, prefix="", cap=8):
    # JSON -> 'a.b.c = value' lines, so a JSON field is searchable as text.
    # `cap` bounds how many array elements are walked: 8 is plenty for
    # searching, but counting leaf fields needs the whole thing.
    lines = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            lines += flatten(v, f"{prefix}.{k}" if prefix else str(k), cap)
    elif isinstance(obj, list):
        for i, v in enumerate(obj[:cap]):
            lines += flatten(v, f"{prefix}[{i}]", cap)
    else:
        lines.append(f"{prefix} = {obj}")
    return lines

GL_HAY = "\n".join([GL_STDOUT, GL_STDERR] + flatten(GL_JSON))
LL_HAY = "\n".join([LB_STDOUT, LB_STDERR, LB_DEVICES, LC_STDOUT, LC_STDERR] +
                   (flatten(LB_JSON) if LB_JSON else []))

# A line that NAMES a metric is not proof the metric was reported. glbench
# prints "energy: not available (RAPL is Linux-only)" -- searching for
# "energy" matches the very line that says there is no energy figure. Without
# this the tables would credit both tools for metrics they explicitly decline
# to produce, which is the opposite of what the row is asking.
NEGATIVE = [r"not available", r"unavailable", r"not reported", r"not measured",
            r"not supported", r"unsupported", r"=\s*(None|null)\s*$", r"\bn/a\b",
            r"does_not_exist", r"not applicable"]

def probe(hay, patterns):
    # Returns (state, evidence): "yes" | "declared-absent" | None.
    # A negated hit does not end the search -- a later match may carry a real
    # value -- but it is remembered in case nothing better turns up.
    negative = None
    for p in patterns:
        for m in re.finditer(p, hay, re.I):
            line = re.sub(r"\s+", " ", m.group(0).replace("\n", " ").strip())[:110]
            if any(re.search(n, line, re.I) for n in NEGATIVE):
                negative = negative or line
                continue
            return ("yes", line)
    return ("declared-absent", negative) if negative else (None, None)

def cell(state, hit, derivable=None):
    if state == "yes":
        return "yes -- `" + hit.replace("|", "\\|") + "`"
    if state == "declared-absent":
        return ("declares it absent -- `" + hit.replace("|", "\\|") + "`")
    if derivable:
        return f"not printed, **derivable** ({derivable})"
    return "not reported"

# metric -> (glbench patterns, llama.cpp patterns, llama derivable-note)
SPECS = {
 "statistical": [
  ("Mean",                  [r"mean[^\n]{0,60}"],                 [r"avg_ts[^\n]{0,40}", r"\bavg\b[^\n]{0,40}"], None),
  ("Median",                [r"median[^\n]{0,60}"],               [r"median[^\n]{0,40}"], "samples_ts in -o json"),
  ("Min / Max",             [r"\bmin\b[^\n]{0,60}", r"\bmax\b[^\n]{0,60}"], [r"\bmin_ts\b[^\n]{0,40}"], "samples_ts in -o json"),
  ("p95",                   [r"p95[^\n]{0,60}"],                  [r"p95[^\n]{0,40}"], "samples_ts in -o json"),
  ("p99",                   [r"p99[^\n]{0,60}"],                  [r"p99[^\n]{0,40}"], "samples_ts in -o json"),
  ("Std deviation",         [r"std_?dev[^\n]{0,60}"],             [r"stddev[^\n]{0,40}"], None),
  ("95% confidence interval",[r"ci95[^\n]{0,60}", r"95%\s*CI[^\n]{0,60}"], [r"ci95[^\n]{0,40}"], "samples_ts + n"),
  ("Cold / warm separated", [r"\bcold\b[^\n]{0,60}", r"warmup[^\n]{0,60}"], [r"warm[- ]?up[^\n]{0,40}"], None),
  ("Iteration count visible",[r"\bcount\b\s*=\s*\d+", r"iters?[^\n]{0,40}"], [r"\breps?\b[^\n]{0,40}", r"samples_ts[^\n]{0,40}"], None),
 ],
 "hardware": [
  ("Device name",           [r"gpu[^\n]{0,80}", r"device[^\n]{0,80}"], [r"CUDA0[^\n]{0,80}", r"Device \d+[^\n]{0,60}"], None),
  ("Peak bandwidth (GB/s)", [r"peak_bandwidth_gbs[^\n]{0,40}"],   [r"bandwidth[^\n]{0,40}"], None),
  ("Measured bandwidth",    [r"read_bandwidth_gbs[^\n]{0,40}", r"observed_bandwidth[^\n]{0,40}"], [r"measured bandwidth[^\n]{0,40}"], None),
  ("Clock speed",           [r"clock[^\n]{0,40}", r"\bMHz\b[^\n]{0,40}"], [r"clock[^\n]{0,40}", r"\bMHz\b[^\n]{0,40}"], None),
  ("ISA flags (AVX2 etc.)", [r"avx2?[^\n]{0,60}", r"isa[^\n]{0,60}"], [r"AVX2?\s*=\s*[01][^\n]{0,60}"], None),
  ("Peak RSS",              [r"peak_memory_bytes[^\n]{0,40}", r"peak rss[^\n]{0,40}"], [r"\bRSS\b[^\n]{0,40}"], None),
  ("CPU utilization %",     [r"cpu_util[^\n]{0,40}", r"utilization[^\n]{0,40}"], [r"cpu ?util[^\n]{0,40}"], None),
  ("Weight size (GB)",      [r"model_bytes[^\n]{0,40}"],          [r"model size[^\n]{0,60}", r"model_size[^\n]{0,40}", r"\d+\.\d+ MiB[^\n]{0,40}"], None),
  ("Parameter count",       [r"param_count[^\n]{0,40}", r"params[^\n]{0,40}"], [r"params[^\n]{0,40}", r"\d+\.\d+ M\b[^\n]{0,30}"], None),
  ("Model load time",       [r"load_ms[^\n]{0,40}", r"load time[^\n]{0,40}"], [r"load time\s*=\s*[^\n]{0,40}"], None),
  ("VRAM usage",            [r"total_memory_bytes[^\n]{0,40}", r"vram[^\n]{0,40}"], [r"buffer size[^\n]{0,60}", r"MiB\s*free[^\n]{0,40}"], None),
  ("Roofline efficiency %", [r"ceiling_efficiency[^\n]{0,40}", r"% of ceiling[^\n]{0,40}"], [r"roofline[^\n]{0,40}", r"efficiency[^\n]{0,40}"], None),
  ("Bottleneck verdict",    [r"bottleneck[^\n]{0,60}"],           [r"bottleneck[^\n]{0,40}"], None),
  ("Energy (RAPL)",         [r"energy[^\n]{0,60}"],               [r"energy[^\n]{0,40}", r"joule[^\n]{0,40}"], None),
 ],
 "behavioral": [
  ("1-gram repetition",     [r"unique_1gram_ratio[^\n]{0,40}"],   [r"1.?gram[^\n]{0,40}"], "raw text is printed"),
  ("2-gram repetition",     [r"unique_2gram_ratio[^\n]{0,40}"],   [r"2.?gram[^\n]{0,40}"], "raw text is printed"),
  ("3-gram repetition",     [r"unique_3gram_ratio[^\n]{0,40}"],   [r"3.?gram[^\n]{0,40}"], "raw text is printed"),
  ("Max repetition run",    [r"max_token_run[^\n]{0,40}"],        [r"max.{0,6}run[^\n]{0,40}"], "raw text is printed"),
  ("Token entropy",         [r"mean_nats[^\n]{0,40}", r"entropy[^\n]{0,40}"], [r"entropy[^\n]{0,40}"], None),
  ("Inter-token stall",     [r"stall[^\n]{0,40}"],                [r"stall[^\n]{0,40}"], None),
  ("Degeneracy flag",       [r"looks_degenerate[^\n]{0,40}"],     [r"degenerate[^\n]{0,40}"], None),
  ("Validation pass/fail",  [r"validation[^\n]{0,60}"],           [r"validation[^\n]{0,40}"], None),
 ],
}

RESULTS = {}
for section, rows in SPECS.items():
    RESULTS[section] = []
    for name, glp, llp, deriv in rows:
        gs, gh = probe(GL_HAY, glp)
        ls, lh = probe(LL_HAY, llp)
        RESULTS[section].append((name, gs, gh, ls, lh, deriv, glp, llp))

def mark(state, deriv=None):
    return {"yes": "Y", "declared-absent": "~"}.get(state, "d" if deriv else ".")

print("  Y = reported   ~ = tool declares it absent   d = derivable   . = nothing found")
for section in SPECS:
    print(f"\n=== {section} ===")
    for name, gs, gh, ls, lh, deriv, _, _ in RESULTS[section]:
        print(f"  {name:26s} glbench={mark(gs)}  llama.cpp={mark(ls, deriv)}")


### Throughput extraction

glbench reports prefill and decode as full `Stats`. llama-bench reports `pp`
(prompt processing) and `tg` (token generation); the mapping is
**pp -> prefill**, **tg -> decode**, stated here rather than assumed.

In [ ]:
def stats_of(node):
    if not isinstance(node, dict):
        return {}
    return {k: node.get(k) for k in
            ("mean", "median", "min", "max", "std_dev", "p95", "p99", "ci95", "count")}

GL_AN = GL_JSON.get("analysis") or {}
GL_PRE = stats_of(GL_AN.get("prefill_tps"))
GL_DEC = stats_of(GL_AN.get("decode_tps"))

def lb_row(kind):
    # kind 'pp' -> the prompt-processing row, 'tg' -> the generation row.
    if not LB_JSON:
        return None
    for r in LB_JSON:
        np_, ng_ = int(r.get("n_prompt", 0) or 0), int(r.get("n_gen", 0) or 0)
        if kind == "pp" and np_ > 0 and ng_ == 0:
            return r
        if kind == "tg" and ng_ > 0 and np_ == 0:
            return r
    return LB_JSON[0] if LB_JSON else None

def lb_stats(r):
    if not r:
        return {}
    s = [float(x) for x in (r.get("samples_ts") or []) if x is not None]
    out = {"mean": r.get("avg_ts"), "std_dev": r.get("stddev_ts"), "count": len(s) or None}
    if s:
        s.sort()
        n = len(s)
        out["median"] = s[n // 2] if n % 2 else (s[n // 2 - 1] + s[n // 2]) / 2
        out["min"], out["max"] = s[0], s[-1]
        out["derived"] = True
    return out

LB_PP, LB_TG = lb_stats(lb_row("pp")), lb_stats(lb_row("tg"))

def fmt(v):
    if v is None:
        return "not reported"
    if isinstance(v, float):
        return f"{v:.1f}"
    return str(v)

def delta(a, b):
    try:
        if a and b:
            return f"{(float(a)/float(b) - 1.0)*100:+.1f}%"
    except Exception:
        pass
    return "--"

THROUGHPUT = [
    ("Prefill mean tok/s",   GL_PRE.get("mean"),   LB_PP.get("mean")),
    ("Prefill median tok/s", GL_PRE.get("median"), LB_PP.get("median")),
    ("Prefill std dev",      GL_PRE.get("std_dev"), LB_PP.get("std_dev")),
    ("Decode mean tok/s",    GL_DEC.get("mean"),   LB_TG.get("mean")),
    ("Decode median tok/s",  GL_DEC.get("median"), LB_TG.get("median")),
    ("Decode std dev",       GL_DEC.get("std_dev"), LB_TG.get("std_dev")),
    ("Iterations",           GL_DEC.get("count"),  LB_TG.get("count")),
]

for n, a, b in THROUGHPUT:
    print(f"{n:24s} glbench={fmt(a):>12s}   llama.cpp={fmt(b):>12s}   {delta(a,b)}")

if LB_PP.get("derived") or LB_TG.get("derived"):
    print("\nnote: llama.cpp medians/min/max are DERIVED from samples_ts (D4),")
    print("      not printed by the tool itself.")


### Write `BENCHMARK_COMPARISON.md`

In [ ]:
NL = "\n"
B = []
def w(s=""):
    B.append(s)

w("# glbench vs llama.cpp — benchmark reporting comparison")
w()
w("Generated by `notebooks/glbench_vs_llamacpp.ipynb`. Every table cell below "
  "is filled from the captured output of the run it describes; nothing is "
  "filled from documentation or memory.")
w()

# ---- Section 1 ------------------------------------------------------------
w("## Section 1: Run conditions")
w()
w(f"- **Date (UTC):** {RUN_META['date_utc']}")
w(f"- **GPU:** {RUN_META['gpu']}")
if RUN_META.get("gpu_count", 1) > 1:
    w(f"- **⚠️ {RUN_META['gpu_count']} GPUs present; both tools were pinned to "
      f"device 0** via `CUDA_VISIBLE_DEVICES=0`. llama.cpp splits across every "
      f"visible device by default — an unpinned run put compute buffers on "
      f"both CUDA0 and CUDA1 — while glbench's glcuda uses one. Without this "
      f"the table below would compare one GPU against two. llama.cpp's "
      f"multi-GPU support is a genuine capability glbench lacks; it is "
      f"credited in Section 7 instead of being folded into the throughput "
      f"numbers.")
w(f"- **Host:** {RUN_META['host']}")
w(f"- **Model:** `{os.path.basename(MODEL_PATH)}` — "
  f"{MODEL_BYTES/1e9:.3f} GB on disk, Qwen2.5-0.5B-Instruct Q4_K_M")
w(f"- **Generation tokens:** {GEN_TOKENS} · **Iterations:** {ITERS} · "
  f"**Warmup:** {WARMUP}")
w(f"- **glbench:** commit `{GL_COMMIT}`, ceiling fix "
  f"{'present' if CEILING_FIX else 'MISSING'}")
w(f"- **llama.cpp:** commit `{LLAMA_COMMIT}`, CUDA backend "
  f"{'confirmed by a real run' if LLAMA_CUDA else 'ABSENT'}, "
  f"built `-j{BUILD_JOBS}` for `sm_{CUDA_ARCH}` (D1); "
  f"`llama-cli` ran in {LC_CONV_MODE}")
w(f"- **llama.cpp build strategy:** {CONFIG_STRATEGY}")
w(f"- **llama.cpp build time:** {LLAMA_BUILD_SECS/60:.1f} min "
  f"({'cold' if LLAMA_COLD else 'warm'}, `-j{BUILD_JOBS}`)")
if not HAVE_CLI:
    w("- ⚠️ **`llama-cli` was not built.** Section 5 and the tokenizer "
      "cross-check are unavailable for llama.cpp in this run — treat their "
      "rows as unmeasured, not as evidence the tool lacks the capability.")
w()
if LLAMA_BUILD_NOTES:
    w("> **How llama.cpp was built matters for the numbers below.**")
    w(">")
    for n in LLAMA_BUILD_NOTES:
        w("> - " + n)
    w()
w("### The prompt")
w()
w(f"Extracted from `glbench/src/main.rs::default_prompt()` — a base sentence "
  f"repeated 8x, then trimmed. {len(PROMPT_TEXT)} characters.")
w()
w("```")
w(PROMPT_TEXT)
w("```")
w()
w(f"- **glbench tokenizer:** {PROMPT_TOKENS} tokens "
  f"(`measurements.iterations[0].prompt_tokens`, not an estimate)")
w(f"- **llama.cpp tokenizer:** "
  f"{LLAMA_PROMPT_TOKENS if LLAMA_PROMPT_TOKENS is not None else 'not found in --verbose-prompt output'}")
if LLAMA_PROMPT_TOKENS is not None and LLAMA_PROMPT_TOKENS != PROMPT_TOKENS:
    w()
    w(f"> ⚠️ The two tokenizers disagree on identical text "
      f"({PROMPT_TOKENS} vs {LLAMA_PROMPT_TOKENS}). Any per-token rate below "
      f"is therefore per-*that-tool's*-token. This is a finding, not a defect "
      f"of the comparison.")
w()
w("### D2 — the prompt is length-matched, not text-matched")
w()
w(f"`llama-bench -p N` takes a token **count** and synthesises a prompt of "
  f"that length; it cannot be handed text. So `llama-bench` ran with "
  f"`-p {PROMPT_TOKENS}` (same length, synthetic content) and only "
  f"`llama-cli` received the exact string. **The throughput table compares "
  f"equal-length work, not identical work.** Prompt *content* does not affect "
  f"prefill cost in a dense transformer, so this is a fair comparison for "
  f"throughput — but it is not the same thing as an identical run, and the "
  f"behavioral section deliberately uses `llama-cli` instead.")
w()

# ---- Section 2 ------------------------------------------------------------
w("## Section 2: Throughput (apples to apples)")
w()
w("Mapping, stated rather than assumed: llama-bench **pp** (prompt processing) "
  "-> **prefill**; **tg** (token generation) -> **decode**.")
w()
w("| Metric | glbench (glcuda) | llama.cpp (CUDA) | Delta |")
w("|---|---|---|---|")
for n, a, b in THROUGHPUT:
    w(f"| {n} | {fmt(a)} | {fmt(b)} | {delta(a, b)} |")
w()
if LB_PP.get("derived") or LB_TG.get("derived"):
    w("llama.cpp's median / min / max are **derived** from the `samples_ts` "
      "array in `-o json` (D4). The tool does not print them; it does print "
      "the raw samples they come from, which is a meaningfully different "
      "answer from 'not available'.")
    w()
w(f"Wall clock: glbench {GL_WALL/60:.1f} min, llama-bench {LB_WALL/60:.1f} min. "
  f"These are not comparable as 'speed' — the tools do different amounts of "
  f"work per invocation (glbench also probes the environment, traces behavior "
  f"and writes an archive).")
w()

# ---- Sections 3-5 ---------------------------------------------------------
SECTION_TITLES = {
    "statistical": ("Section 3: Statistical rigor", None),
    "hardware":    ("Section 4: Hardware insight", None),
    "behavioral":  ("Section 5: Behavioral analysis", None),
}
for key, (title, _) in SECTION_TITLES.items():
    w(f"## {title}")
    w()
    w("| Metric | glbench | llama.cpp |")
    w("|---|---|---|")
    for name, gs, gh, ls, lh, deriv, _, _ in RESULTS[key]:
        w(f"| {name} | {cell(gs, gh)} | {cell(ls, lh, deriv)} |")
    w()

w(f"**Warmup.** glbench ran {WARMUP} warmup iterations and reports them apart "
  f"from the {ITERS} measured ones. llama-bench "
  + ("also warms up by default — it documents `--no-warmup` to switch that off "
     "— but does not report the warmup separately from the measured "
     "repetitions. The row above is therefore about the *reporting*, not the "
     "behaviour: both tools warm up, one of them tells you."
     if LB_WARMS_UP else
     "shows no warmup flag in its documented options."))
w()
w("### How these three tables were filled")
w()
w("Each row is a regex search over that tool's **maximum-verbosity output** — "
  "terminal text plus the flattened JSON it emits. A cell reads *not reported* "
  "only after the search missed. The patterns are listed below so a reader can "
  "disagree with them; a metric a tool names something unanticipated would show "
  "as a false negative, and that is a genuine weakness of this method.")
w()
w("Three of the four possible answers are distinct on purpose:")
w()
w("- **yes** — a line carrying an actual value was found.")
w("- **declares it absent** — the tool named the metric and said it had no "
  "value. Searching for `energy` otherwise matches glbench's own "
  "*\"energy: not available\"* line and credits it with a measurement it "
  "explicitly refused to fabricate. Declaring an absence is a design choice "
  "worth distinguishing from silence, but it is not a reported number.")
w("- **derivable** — not printed, but computable from what the tool does "
  "print (D4).")
w("- **not reported** — the search found nothing at all.")
w()
w("<details><summary>Search patterns used</summary>")
w()
w("| Metric | glbench patterns | llama.cpp patterns |")
w("|---|---|---|")
for key in SPECS:
    for name, _, _, _, _, _, glp, llp in RESULTS[key]:
        gp = ", ".join("`" + p.replace("|", "\\|") + "`" for p in glp)
        lp = ", ".join("`" + p.replace("|", "\\|") + "`" for p in llp)
        w(f"| {name} | {gp} | {lp} |")
w()
w("</details>")
w()

# ---- Section 6 ------------------------------------------------------------
w("## Section 6: Output format")
w()
w("| Aspect | glbench | llama.cpp |")
w("|---|---|---|")
w(f"| Terminal output | structured sections (environment, measurements, "
  f"analysis, hypotheses) | flat table (`llama-bench`) / interleaved log "
  f"(`llama-cli`) |")
w(f"| Machine-readable | {len(flatten(GL_JSON, cap=10**6))} leaf fields in a "
  f"versioned envelope | {len(flatten(LB_JSON, cap=10**6)) if LB_JSON else 0} "
  f"leaf fields, flat array of rows |")
_null_declared = len((GL_JSON.get("availability") or {}))
w(f"| Absent fields | every `null` carries a declared reason "
  f"({_null_declared} declarations in this run) | absent fields are simply "
  f"absent |")
w(f"| Caveats surfaced | {len((GL_AN.get('notes') or []))} notes, "
  f"{len((GL_AN.get('hypotheses') or []))} hypotheses phrased as "
  f"*consistent with*, never as causes | none |")
w(f"| Self-diagnosis | warns on noisy runs, implausible ceilings, and "
  f"unmeasurable counters | no |")
w()
w("A note on the JSON field counts: more fields is not automatically better. "
  "llama-bench's flat array is trivially loadable into a dataframe and its "
  "schema fits in your head. glbench's envelope carries provenance for every "
  "figure and costs you a nested walk to read.")
w()

# ---- Section 7 ------------------------------------------------------------
w("## Section 7: Verdict")
w()
ALL_ROWS = RESULTS["statistical"] + RESULTS["hardware"] + RESULTS["behavioral"]

# "Only" means the other tool printed no value. A metric the other tool can
# still be made to yield is listed with that caveat rather than claimed
# outright -- otherwise glbench gets credit for a median llama.cpp hands you
# the samples for.
gl_only = [(n, d) for n, gs, gh, ls, lh, d, _, _ in ALL_ROWS
           if gs == "yes" and ls != "yes"]
ll_only = [(n, gs) for n, gs, gh, ls, lh, d, _, _ in ALL_ROWS
           if ls == "yes" and gs != "yes"]
both_missing = [n for n, gs, gh, ls, lh, d, _, _ in ALL_ROWS
                if gs != "yes" and ls != "yes"]

w("### What glbench tells you that llama.cpp does not")
w()
if gl_only:
    for n, d in gl_only:
        w(f"- **{n}**" + (f" — though llama.cpp makes it derivable ({d})."
                          if d else "."))
else:
    w("- *(nothing in this run)*")
w()
w("### Where llama.cpp is ahead")
w()
w("This section is mandatory and is not a formality.")
w()
if ll_only:
    for n, gs in ll_only:
        w(f"- **{n}** — llama.cpp prints a value; glbench "
          + ("declares it absent rather than measuring it."
             if gs == "declared-absent" else "does not report it at all."))
else:
    w("- No metric in the probed set was reported by llama.cpp and missing "
      "from glbench.")
w()
w("Beyond the table, four advantages that a metric checklist cannot show:")
w()
w(f"- **It builds and runs everywhere.** llama.cpp targets CPU, CUDA, Metal, "
  f"Vulkan, ROCm, SYCL. glbench's GPU path is glcuda alone; on any other "
  f"accelerator this comparison could not have been run at all.")
if RUN_META.get("gpu_count", 1) > 1:
    w(f"- **It uses hardware glbench cannot reach.** Given "
      f"{RUN_META['gpu_count']} GPUs, llama.cpp split the model across all of "
      f"them without being asked; glbench ran on one. That capability had to "
      f"be switched off to make Section 2 a fair test, which is itself the "
      f"finding: the fair comparison is the one that handicaps llama.cpp.")
w(f"- **Its numbers are the reference.** llama-bench is what the field quotes. "
  f"A glbench figure has to be argued for; a llama-bench figure is understood "
  f"on sight.")
w(f"- **The flat schema is a feature.** `-o json` gives an array of rows that "
  f"drops into a dataframe. glbench's envelope is richer and correspondingly "
  f"more work to consume.")
w(f"- **Sweeps are first-class.** `llama-bench` varies batch size, thread "
  f"count, ngl and quantisation in one invocation. glbench has `scale --sweep` "
  f"but it covers less ground.")
w()
w("### What this run would have missed with only one tool")
w()
_eff = GL_AN.get("ceiling_efficiency")
_bn  = GL_AN.get("bottleneck")
w(f"**With llama.cpp alone:** the throughput numbers, and nothing about "
  f"whether they are *good*. This run's decode sits at "
  f"{('%.0f%%' % (_eff*100)) if isinstance(_eff, (int, float)) else 'an undetermined fraction'} "
  f"of the bandwidth ceiling with a verdict of `{_bn}` — the question "
  f"\"is 0.5B too small to saturate this card?\" is answerable from glbench's "
  f"output and unanswerable from llama-bench's.")
w()
w(f"**With glbench alone:** no cross-check. llama.cpp independently "
  f"re-tokenized the identical prompt"
  + (f" and agreed at {PROMPT_TOKENS} tokens"
     if LLAMA_PROMPT_TOKENS == PROMPT_TOKENS
     else f" and disagreed ({PROMPT_TOKENS} vs {LLAMA_PROMPT_TOKENS})")
  + ", and a second independent implementation of the same model on the same "
    "hardware is the only thing that can tell you whether a glbench number is "
    "the engine's property or the benchmark's.")
w()
_declared = [n for n, gs, gh, ls, lh, d, _, _ in ALL_ROWS if gs == "declared-absent"]
_silent = [n for n in both_missing if n not in _declared]
if _silent:
    w("**Neither tool produced a value for:** " + ", ".join(_silent) + ".")
    w()
if _declared:
    w("**glbench named but declined to produce:** " + ", ".join(_declared) +
      ". Worth separating from the line above — a declared absence tells you "
      "the counter was looked for and the platform could not supply it, which "
      "is a different fact from a metric nobody attempted.")
    w()
w("### Which tool for which question")
w()
w("- *\"How fast is this build?\"* — llama.cpp. Faster to get, universally "
  "understood, sweeps configurations.")
w("- *\"Why is it that fast, and is that as fast as this hardware allows?\"* — "
  "glbench. Ceiling, bottleneck, per-stage roofline, and a declared reason for "
  "every figure it could not produce.")
w("- *\"Is the output any good?\"* — glbench measures repetition (and token "
  "entropy when the run is traced); llama.cpp prints the text and leaves the "
  "judgement to you. Note that llama.cpp printing the raw text is not nothing: "
  "it is the input those statistics are computed from.")
w()

# ---- Appendix -------------------------------------------------------------
w("## Appendix: raw output")
w()
# Fold runs of near-identical lines into one line plus a count.
#
# `-v` makes llama.cpp emit a line per CUDA graph reuse -- 1280 of them in an
# observed run, alternating between two graph ids -- which would fill the
# entire appendix with one repeated sentence and push the real output past the
# size cap. Digits are normalised before comparing so the alternating ids fold
# together. The fold is stated inline, so nothing is silently dropped: every
# distinct line survives and the count is itself the information.
def collapse(text, threshold=4):
    lines = (text or "").splitlines()
    out, i = [], 0
    while i < len(lines):
        key = re.sub(r"\d+", "#", lines[i])
        j = i + 1
        while j < len(lines) and re.sub(r"\d+", "#", lines[j]) == key:
            j += 1
        n = j - i
        if n >= threshold:
            out.append(lines[i])
            out.append(f"    ... [{n - 1} more lines like the one above, folded]")
        else:
            out.extend(lines[i:j])
        i = j
    return "\n".join(out)

for title, body in [
    ("glbench — terminal", GL_STDOUT),
    ("llama-bench — stdout (JSON)", LB_STDOUT),
    ("llama-bench — stderr (verbose, repeated lines folded)", collapse(LB_STDERR)),
    ("llama-bench — --list-devices", LB_DEVICES),
    ("llama-cli — verbose (repeated lines folded)", collapse(LC_STDERR or LC_STDOUT)),
]:
    w(f"### {title}")
    w()
    w("```")
    w((body or "(empty)").strip()[:20000])
    w("```")
    w()

MD_PATH = os.path.join(OUT_DIR, "BENCHMARK_COMPARISON.md")
open(MD_PATH, "w", encoding="utf-8").write(NL.join(B))
print(f"wrote {MD_PATH}  ({len(NL.join(B))} chars)")
print(f"also: glbench_result.json, llama_result.json, glbench_output.txt, "
      f"llama_output.txt, llama_cli_verbose.txt")


## Step 7 — Print Section 2 and Section 7

In [ ]:
text = open(os.path.join(OUT_DIR, "BENCHMARK_COMPARISON.md"), encoding="utf-8").read()

def section(name, upto):
    i = text.find(f"## {name}")
    j = text.find(f"## {upto}", i + 1) if upto else len(text)
    return text[i:j if j > 0 else len(text)]

print(section("Section 2", "Section 3"))
print(section("Section 7", "Appendix"))
